In [1]:
import torch
from transformers import LlamaForCausalLM, LlamaTokenizer;
from llama_models.injected_llama_for_causal import LlamaForCausalLM as InjectedLlama
from transformers import (
    LlamaConfig as HFConfig,
    AutoTokenizer
)
import yaml
from utils.data_utils import Struct

/home/huang717/.conda/envs/test/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = "/home/huang717/.llama/checkpoints/Llama-2-7b"
model_name="meta-llama/Llama-2-7b-hf"
config_path = "/home/huang717/DRAGN/IRM/injectable-alignment-model/configs/Llama-2-7b-chat-hf_tiny_shakespeare_31_training.yaml"
with open(config_path, "r") as f:
        config = yaml.safe_load(f)

config = Struct(**config)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 100%|██████████| 776/776 [00:00<00:00, 3.39MB/s]
tokenizer.model: 100%|██████████| 500k/500k [00:00<00:00, 4.00MB/s]
tokenizer.json: 100%|██████████| 1.84M/1.84M [00:00<00:00, 12.5MB/s]
special_tokens_map.json: 100%|██████████| 414/414 [00:00<00:00, 1.94MB/s]


In [ ]:
injected_model = InjectedLlama(tokenizer, config)

In [ ]:
print(injected_model)

In [3]:
vanilla_model = LlamaForCausalLM.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:44<00:00, 22.36s/it]


In [4]:
# Basic save
torch.save(vanilla_model.state_dict(), '/home/huang717/DRAGN/IRM/injectable-alignment-model/default_checkpoints/Llama-2-7b-hf.ckpt')

In [ ]:
print(vanilla_model)

In [ ]:
# Prepare input
input_text = "Once upon a time"
inputs = tokenizer(input_text, return_tensors="pt")

# Generate
outputs = vanilla_model.generate(
    inputs.input_ids,
    max_length=50,  # maximum length of generated text
    num_return_sequences=1,  # number of sequences to generate
    temperature=0.7,  # controls randomness (lower = more deterministic)
    do_sample=False,  # use sampling instead of greedy decoding
)

# Decode the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
print(vanilla_model.model.layers[0].self_attn.q_proj.weight)

In [ ]:
vanilla_model.model.layers[0].self_attn.q_proj.weight = torch.nn.Parameter(
    torch.zeros_like(vanilla_model.model.layers[0].self_attn.q_proj.weight)
)
print(vanilla_model.model.layers[0].self_attn.q_proj.weight)

In [ ]:
# Prepare input
input_text = "Once upon a time"
inputs = tokenizer(input_text, return_tensors="pt")

# Generate
outputs = vanilla_model.generate(
    inputs.input_ids,
    max_length=50,  # maximum length of generated text
    num_return_sequences=1,  # number of sequences to generate
    temperature=0.7,  # controls randomness (lower = more deterministic)
    do_sample=False,  # use sampling instead of greedy decoding
)

# Decode the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

In [10]:
original_checkpoint_path = "/home/huang717/DRAGN/IRM/injectable-alignment-model/default_checkpoints/Llama-2-7b-chat-hf.ckpt"
checkpoint = torch.load(original_checkpoint_path,  map_location=torch.device('cpu'))


In [ ]:
vanilla_model.load_state_dict(checkpoint['state_dict'], strict=False)

In [ ]:
print(vanilla_model.model.layers[0].self_attn.q_proj.weight)

In [ ]:
# Prepare input
input_text = "Once upon a time"
inputs = tokenizer(input_text, return_tensors="pt")

# Generate
outputs = vanilla_model.generate(
    inputs.input_ids,
    max_length=50,  # maximum length of generated text
    num_return_sequences=1,  # number of sequences to generate
    temperature=0.7,  # controls randomness (lower = more deterministic)
    do_sample=False,  # use sampling instead of greedy decoding
)

# Decode the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
the_model =  InjectedLlama.from_pretrained(model_name, irm_config=config)